# End-to-End ML Classification: Census Income Prediction (v2 — Portfolio Clean Version)

**Author:** Chiemela Joseph Nwosu  
**Version:** v2 — Portfolio-ready, leakage-corrected pipeline  
**Original notebook:** `E2E_ML_Classification_Model.ipynb`
---

## About This Notebook

This is a cleaned, portfolio-ready version of the original classification project. It addresses two documented limitations from the original notebook:

1. **SMOTE applied before train/test split** — In the original notebook, SMOTE was applied to the full dataset before splitting. This caused synthetic minority-class samples to appear in both training and test sets, inflating evaluation metrics. This version applies SMOTE **only to the training data**, after the split.

2. **Potential target leakage via `income_1`** — The original preprocessing pipeline applied one-hot encoding to all categorical columns including the target `income`, creating an `income_1` column. This version explicitly drops any target-derived columns from the feature matrix before training.

The original notebook is preserved unchanged as a learning artifact from *Mastering ChatGPT and Google Colab for Machine Learning* by Moscato.

---

## Workflow Overview

| Step | Description |
|---|---|
| 1 | Load raw dataset from `data/raw/Census.csv` |
| 2 | Exploratory data analysis |
| 3 | Preprocessing: encoding + standardization |
| 4 | Train/test split (70/30) |
| 5 | SMOTE applied to training data only |
| 6 | Model training: Logistic Regression, Random Forest, SVM |
| 7 | Evaluation: accuracy, precision, recall, F1, confusion matrix |
| 8 | Hyperparameter tuning with GridSearchCV |
| 9 | Feature importance analysis |
| 10 | Export visuals to `visuals/` folder |

## Section 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create visuals directory if it doesn't exist
os.makedirs('../visuals', exist_ok=True)

# Load raw dataset
# NOTE: This notebook reads from data/raw/Census.csv (relative to repo root).
# If running from the notebooks/ folder, use the path below.
# If running from repo root, change to 'data/raw/Census.csv'.
DATA_PATH = '../data/raw/Census.csv'
VISUALS_PATH = '../visuals/'

df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## Section 2: Exploratory Data Analysis

In [ ]:
# Basic info
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nMissing values (? as placeholder):')
print((df == ' ?').sum())

In [ ]:
# Class distribution
income_counts = df['income'].value_counts()
print('Income class distribution:')
print(income_counts)
print(f'\nClass balance: {income_counts.values[0]/len(df)*100:.1f}% / {income_counts.values[1]/len(df)*100:.1f}%')

fig, ax = plt.subplots(figsize=(7, 4))
income_counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='black')
ax.set_title('Income Class Distribution (Original Dataset)', fontsize=13)
ax.set_xlabel('Income Class')
ax.set_ylabel('Count')
ax.set_xticklabels(income_counts.index, rotation=0)
for i, v in enumerate(income_counts.values):
    ax.text(i, v + 100, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(VISUALS_PATH + 'class_distribution.png', dpi=150)
plt.show()
print('Saved: visuals/class_distribution.png')

In [ ]:
# Numerical feature distributions
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numerical columns:', numerical_cols)
df[numerical_cols].describe()

## Section 3: Preprocessing

### Why This Differs from the Original Notebook

The original notebook applied the full preprocessing pipeline (including SMOTE) to the entire dataset before splitting. This v2 notebook separates preprocessing into two phases:

**Phase 1 (before split):** Encoding and cleaning only — no SMOTE, no scaling applied to the full dataset.

**Phase 2 (after split):** SMOTE applied to training data only. StandardScaler fit on training data only, then applied to both train and test.

### Target Leakage Fix

The original pipeline applied one-hot encoding to all categorical columns including `income`, creating an `income_1` column. Including this in the feature matrix `X` would be direct target leakage — the model would be learning from a transformed version of the answer. This version explicitly removes any target-derived columns before training.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Work on a copy
df_clean = df.copy()

# Replace ' ?' with NaN and drop those rows
df_clean = df_clean.replace(' ?', np.nan).dropna()
print(f'Rows after dropping missing values: {len(df_clean)}')

# Strip whitespace from string columns
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()

# Encode target variable: <=50K -> 0, >50K -> 1
df_clean['income'] = (df_clean['income'] == '>50K').astype(int)
print('Target encoding: <=50K=0, >50K=1')
print(df_clean['income'].value_counts())

In [ ]:
# Separate features and target BEFORE any encoding that could leak the target
y = df_clean['income'].copy()
X_raw = df_clean.drop(columns=['income']).copy()

print('Feature matrix shape:', X_raw.shape)
print('Target shape:', y.shape)
print('\nFeature columns:')
print(X_raw.columns.tolist())

In [ ]:
# One-hot encode categorical features (target is already separated — no leakage possible)
X_encoded = pd.get_dummies(X_raw, drop_first=True)

print('Encoded feature matrix shape:', X_encoded.shape)

# Verify no target-derived columns are present
target_leak_cols = [c for c in X_encoded.columns if 'income' in c.lower()]
if target_leak_cols:
    print(f'WARNING: Target-derived columns found and will be dropped: {target_leak_cols}')
    X_encoded = X_encoded.drop(columns=target_leak_cols)
else:
    print('No target-derived columns found in feature matrix. No leakage.')

print('Final feature matrix shape:', X_encoded.shape)

## Section 4: Train/Test Split

The train/test split is performed **before** SMOTE. This ensures the test set contains only real, original data — not synthetic samples. Evaluating on synthetic test data inflates metrics and gives a misleading picture of real-world performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

print('Train/test split (70/30, stratified):')
print(f'  Training samples: {X_train.shape[0]}')
print(f'  Test samples:     {X_test.shape[0]}')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

## Section 5: SMOTE on Training Data Only

SMOTE (Synthetic Minority Oversampling Technique) generates synthetic samples for the minority class to address class imbalance.

**Critical rule:** SMOTE must only be applied to the training set. Applying it before the split contaminates the test set with synthetic data, making evaluation metrics unreliable. The test set must always reflect the real-world distribution.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('After SMOTE (training data only):')
print(f'  Training samples: {X_train_resampled.shape[0]}')
print(f'  Class distribution:')
print(pd.Series(y_train_resampled).value_counts())
print(f'\nTest set unchanged: {X_test.shape[0]} samples (original data only)')

## Section 6: Feature Scaling

StandardScaler is fit **only on training data** and then applied to both train and test sets. Fitting the scaler on the full dataset before splitting would leak test set statistics into the training process.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data only, transform both
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

print('Scaling complete.')
print(f'  Training shape: {X_train_scaled.shape}')
print(f'  Test shape:     {X_test_scaled.shape}')

## Section 7: Model Training and Evaluation

Three classifiers are trained and evaluated using accuracy, precision, recall, and F1-score. Because the test set contains only original (non-synthetic) data, these metrics reflect realistic performance estimates.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred)
    }
    trained_models[name] = model

results_df = pd.DataFrame(results).T
print('Model Comparison (evaluated on original test data):')
print(results_df.round(4))

# TODO: Run all cells to populate these results after correcting the pipeline.

## Section 8: Confusion Matrix

The confusion matrix shows the breakdown of true positives, true negatives, false positives, and false negatives for the best-performing model.

In [ ]:
# Use Logistic Regression for confusion matrix (interpretable baseline)
best_model_name = 'Logistic Regression'
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['<=50K', '>50K'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_model_name}\n(v2: SMOTE on train only, original test set)', fontsize=11)
plt.tight_layout()
plt.savefig(VISUALS_PATH + 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved: visuals/confusion_matrix.png')

print(f'\nClassification Report — {best_model_name}:')
print(classification_report(y_test, y_pred_best, target_names=['<=50K', '>50K']))

# TODO: Run all cells to generate and save this chart.

## Section 9: ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.decision_function(X_test_scaled)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models\n(v2: evaluated on original test set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(VISUALS_PATH + 'roc_curve.png', dpi=150)
plt.show()
print('Saved: visuals/roc_curve.png')

# TODO: Run all cells to generate and save this chart.

## Section 10: Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Cross-validate on the SMOTE-resampled training data
# Note: for a fully rigorous CV, SMOTE should be inside the CV pipeline.
# This is a simplified version for portfolio demonstration.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-Fold Cross-Validation (on SMOTE-resampled training data):')
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train_resampled, cv=cv, scoring='f1')
    print(f'  {name}: F1 = {scores.mean():.4f} (+/- {scores.std():.4f})')

# TODO: Run all cells to populate these results.

## Section 11: Hyperparameter Tuning (Logistic Regression)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [500]
}

lr = LogisticRegression(random_state=42)
grid_search = GridSearchCV(lr, param_grid, cv=5, scoring='f1', verbose=1, n_jobs=-1)
grid_search.fit(X_train_scaled, y_train_resampled)

print('Best parameters:', grid_search.best_params_)
print('Best CV F1 score:', round(grid_search.best_score_, 4))

# Evaluate tuned model on test set
tuned_lr = grid_search.best_estimator_
y_pred_tuned = tuned_lr.predict(X_test_scaled)

print('\nTuned Logistic Regression — Test Set Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_tuned):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_tuned):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_tuned):.4f}')

# TODO: Run all cells to populate these results.

## Section 12: Feature Importance

In [ ]:
# Feature importance from tuned Logistic Regression coefficients
feature_names = X_encoded.columns.tolist()
coef = np.abs(tuned_lr.coef_[0])
importance_df = pd.DataFrame({'feature': feature_names, 'importance': coef})
importance_df = importance_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='#4C72B0')
ax.set_xlabel('Absolute Coefficient Value')
ax.set_title('Top 20 Feature Importances — Logistic Regression (v2)', fontsize=12)
plt.tight_layout()
plt.savefig(VISUALS_PATH + 'feature_importance.png', dpi=150)
plt.show()
print('Saved: visuals/feature_importance.png')

print('\nTop 10 features:')
print(importance_df.head(10).to_string(index=False))

# TODO: Run all cells to generate and save this chart.

## Section 13: Model Persistence

In [ ]:
import joblib

# Save the tuned logistic regression model
model_path = '../logistic_regression_model_v2.pkl'
joblib.dump(tuned_lr, model_path)
print(f'Model saved to: {model_path}')

# Quick prediction test
sample = X_test_scaled[[0]]
loaded_model = joblib.load(model_path)
pred = loaded_model.predict(sample)
label = '>50K' if pred[0] == 1 else '<=50K'
print(f'Sample prediction: {label}')

# TODO: Run all cells to generate the saved model file.

## Section 14: Summary

### What Changed from v1 to v2

| Issue | Original Notebook (v1) | This Notebook (v2) |
|---|---|---|
| SMOTE placement | Applied to full dataset before split | Applied to training data only, after split |
| Target leakage | `income_1` column potentially in X | Target separated before encoding; leakage check added |
| Scaler fit | Not explicitly controlled | Fit on training data only |
| Test set | May contain synthetic samples | Contains only original data |
| Evaluation reliability | Inflated (near-perfect scores) | Realistic estimates on original data |
| Data path | Google Colab upload | `data/raw/Census.csv` (local path) |
| Visual export | Not implemented | Charts saved to `visuals/` folder |

### TODO Before Committing

- [ ] Run all cells in this notebook to generate outputs and saved charts
- [ ] Verify the evaluation metrics look realistic (expected: ~82–87% accuracy)
- [ ] Confirm `visuals/class_distribution.png`, `confusion_matrix.png`, `roc_curve.png`, `feature_importance.png` are saved
- [ ] Update `docs/model_evaluation.md` with the actual v2 metrics after running
- [ ] Commit both the original notebook and this v2 notebook